In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
warnings.filterwarnings('ignore')

btc_price_data = "BTC-USD_price.csv"
spread_raw_data = "hourly_avg_e_spread_2021_09.csv"
event_filter_data = "event_filter_2021_09.csv"

## Functions Preparation

In [ ]:
def load_data(btc_price_data, spread_raw_data, event_filter_data, coin_spread):

    price_data = pd.read_csv(btc_price_data, parse_dates=['datetime'])          # parse means to convert the date column to datetime
    price_data = price_data.sort_values(by='datetime')                          # sort the data by date

    spread_data = pd.read_csv(spread_raw_data, parse_dates=['datetime'])    # parse means to convert the date column to datetime
    spread_data = spread_data.sort_values(by='datetime')                    # sort the data by date
    spread_data[coin_spread] = pd.to_numeric(spread_data[coin_spread], errors='coerce')
    spread_data[coin_spread] = spread_data[coin_spread].fillna(0)

    event_data = pd.read_csv(event_filter_data, parse_dates=['date'])

    return price_data, spread_data, event_data


def detect_event_hour(price_data, spread_data, event_date, n_hours):

    event_day_price = price_data[    # Filter the data for the event day, including previous and next days
        (price_data['datetime'] >= pd.Timestamp(event_date) - pd.Timedelta(hours=n_hours)) &    # include data of previous n_days
        (price_data['datetime'] < pd.Timestamp(event_date) + pd.Timedelta(hours=n_hours))       # include data of next n_days
    ]
    event_day_price['high_low_diff'] = abs(event_day_price['high'] - event_day_price['low'])    # calculate the difference between high and low for each hour
    event_ob_hour = event_day_price.loc[event_day_price['high_low_diff'].idxmax(), 'datetime']  # take the hour with the maximum difference as the event_hour
    event_ob_spread = spread_data.loc[spread_data['datetime'] == event_ob_hour, coin_spread].values[0]  # retrieve the spread value at the event_ob_hour

    return event_ob_hour, event_ob_spread


def calculate_baseline(spread_data, event_ob_hour, m_days):

    baseline_data = spread_data[    # Filter the data for the baseline period
        (spread_data['datetime'] >= pd.Timestamp(event_ob_hour) - pd.Timedelta(hours=24*m_days)) &  # include data btw m_days
        (spread_data['datetime'] < pd.Timestamp(event_ob_hour))                                     # and 1 hour before the event_ob_hour
    ]
    baseline_avg_spread = baseline_data[coin_spread].mean()   # Calculate the average spread

    return baseline_data, baseline_avg_spread


def calculate_diff(spread_data, event_ob_hour, baseline_avg_spread, k_days):

    post_event_data = spread_data[    # Filter the data for the post-event period
        (spread_data['datetime'] >= pd.Timestamp(event_ob_hour)) &   # include date between event_ob_hour
        (spread_data['datetime'] < pd.Timestamp(event_ob_hour) + pd.Timedelta(hours=24*k_days))  # and k_days afterward
    ]
    post_event_data['diff_spread'] = post_event_data[coin_spread] - baseline_avg_spread  # Calculate the difference for each hour

    return post_event_data

## Generate Panel

In [ ]:
def generate_panel_df(price_data, spread_data, event_data, coin_spread, output_folder):

    # Initialize panel_df
    num_rows = m_days * 24 + 1 + k_days * 24   # no. of rows: baseline + event + post-event
    num_cols = len(event_data)                 # no. of columns: number of events
    panel_df = pd.DataFrame(index=range(num_rows), columns=[f"event{i+1}" for i in range(num_cols)])   # Create the empty DataFrame
    panel_df[:] = np.nan                       # Initialize with NaN

    # Iterate over each event
    for i, event_date in enumerate(event_data['date']):
        # Calculate baseline, event hour, and post-event differences
        event_ob_hour, event_ob_spread = detect_event_hour(price_data, spread_data, event_date, n_hours)
        baseline_data, baseline_avg_spread = calculate_baseline(spread_data, event_ob_hour, m_days)
        baseline_spread_array = np.array(baseline_data[coin_spread]) # Convert the spread data to an array
        post_event_data = calculate_diff(spread_data, event_ob_hour, baseline_avg_spread, k_days)
        diff_spread_array = np.array(post_event_data[coin_spread])   # Convert the spread data to an array

        event_array = np.concatenate([            # Combine baseline, event, and post-event data
            baseline_spread_array[:24 * m_days],  # Baseline hours
            np.array([event_ob_spread]),          # Event observation hour
            diff_spread_array[:24 * k_days]       # Post-event hours
        ])
        event_array = event_array.reshape(-1)     # Ensure vertical structure

        column_name = f"event{i+1}"               # Name the column with the event number
        panel_df.loc[:len(event_array) - 1, column_name] = event_array    # Insert the event array into the DataFrame

    # Insert a new column at the beginning of the DataFrame, representing the hour timestamps
    panel_df.insert(0, 'Time', list(range(-m_days * 24, 0)) + [0] + list(range(1, k_days * 24 + 1)))

    # Save the DataFrame as a CSV file to the assigned folder
    os.makedirs(output_folder, exist_ok=True)
    file_path = os.path.join(output_folder, f'abn_spread_panel_df_{coin_spread}.csv')
    panel_df.to_csv(file_path, index=False)
    print(f"Panel saved to {file_path}.")

    return panel_df


## By-type Panels Output

In [ ]:
# Set the parameters
n_hours = 18   # DETERMINED: the event_ob_hour in a range of n_h hours before and after the event outbreak
m_days = 30    # DETERMINED: how many days for calculating the baseline
k_days = 10    # DETERMINED: how many days for calculating the difference
coin_list = ["BTC", "SHIB", "USDT", "DOGE"]

# Regroup the events by binary_type and trinary_type
_, _, event_data = load_data(btc_price_data, spread_raw_data, event_filter_data, "BTC")
binary_groups = event_data.groupby('Binary Type')
trinary_groups = event_data.groupby('Trinary Type')

# Create the binary_type folder
for binary_type, binary_events in binary_groups:
    binary_folder = f"panel_bi-type_{binary_type}"
    for coin_spread in coin_list:
        price_data, spread_data, _ = load_data(btc_price_data, spread_raw_data, event_filter_data, coin_spread)
        generate_panel_df(price_data, spread_data, binary_events, coin_spread, binary_folder)

# Create the trinary_type folder
trinary_mapping = {"Macroeconomy": "tri-type_econ", "Crypto-Regulation": "tri-type_regu", "Crypto-Tech": "tri-type_tech"}
for trinary_type, trinary_events in trinary_groups:
    trinary_folder = f"panel_{trinary_mapping[trinary_type]}"
    for coin_spread in coin_list:
        price_data, spread_data, _ = load_data(btc_price_data, spread_raw_data, event_filter_data, coin_spread)
        generate_panel_df(price_data, spread_data, trinary_events, coin_spread, trinary_folder)

Panel saved to panel_bi-type_0/abn_spread_panel_df_BTC.csv.
Panel saved to panel_bi-type_0/abn_spread_panel_df_SHIB.csv.
Panel saved to panel_bi-type_0/abn_spread_panel_df_USDT.csv.
Panel saved to panel_bi-type_0/abn_spread_panel_df_DOGE.csv.
Panel saved to panel_bi-type_1/abn_spread_panel_df_BTC.csv.
Panel saved to panel_bi-type_1/abn_spread_panel_df_SHIB.csv.
Panel saved to panel_bi-type_1/abn_spread_panel_df_USDT.csv.
Panel saved to panel_bi-type_1/abn_spread_panel_df_DOGE.csv.
Panel saved to panel_tri-type_regu/abn_spread_panel_df_BTC.csv.
Panel saved to panel_tri-type_regu/abn_spread_panel_df_SHIB.csv.
Panel saved to panel_tri-type_regu/abn_spread_panel_df_USDT.csv.
Panel saved to panel_tri-type_regu/abn_spread_panel_df_DOGE.csv.
Panel saved to panel_tri-type_tech/abn_spread_panel_df_BTC.csv.
Panel saved to panel_tri-type_tech/abn_spread_panel_df_SHIB.csv.
Panel saved to panel_tri-type_tech/abn_spread_panel_df_USDT.csv.
Panel saved to panel_tri-type_tech/abn_spread_panel_df_DOGE.c

## Panel Visualization Output
(to be continued)